**Ex 2**

In [59]:
# function to compute cartesian coordinates of a GNSS station, from the cartesian coordinates of the satellite and the geodetic coordinates of the station

import numpy as np

def compute_cartesian_coordinates(station_geodetic):
    
    lat, lon = station_geodetic
    # radians conversion
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    print(f"Original geodetic coordinates: {station_geodetic}")
    print(f"Converted to radians: latitude={lat_rad}, longitude={lon_rad}")
    a = 6378137.0 # Semi-major axis of the Earth in meters
    f = 1 / 298.257223563  # Flattening of the Earth
    e2 = 2 * f - f**2  # Square of the first eccentricity
    print(f"Eccentricity squared: {e2}")
    W = np.sqrt(1 - e2 * np.sin(lat_rad)**2)
    print(f"Computed W: {W}")

    # cartesian coordinates of the station
    x = a * np.cos(lat_rad) * np.cos(lon_rad) / W 
    y = a * np.cos(lat_rad) * np.sin(lon_rad) / W
    z = a * (1 - e2) * np.sin(lat_rad) / W

    return np.array([x, y, z])

In [60]:
def compute_azimuth_elevation(satellite_position, station_geodetic):

    lat, lon = station_geodetic
    # radians conversion
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)

    a = 6378137.0 # Semi-major axis of the Earth in meters
    f = 1 / 298.257223563  # Flattening of the Earth
    e2 = 2 * f - f**2  # Square of the first eccentricity
    print(f"Eccentricity squared: {e2}")
    W = np.sqrt(1 - e2 * np.sin(lat_rad)**2)
    print(f"Computed W: {W}")

    # cartesian coordinates of the station
    x = a * np.cos(lat_rad) * np.cos(lon_rad) / W 
    y = a * np.cos(lat_rad) * np.sin(lon_rad) / W
    z = a * (1 - e2) * np.sin(lat_rad) / W

    # elevation and azimuth of the satellite with respect to the station
    sat_x, sat_y, sat_z = satellite_position
    dx = sat_x - x
    dy = sat_y - y
    dz = sat_z - z

    # rotation matrix
    R = np.array([
        [-np.sin(lon_rad), np.cos(lon_rad), 0],
        [-np.sin(lat_rad) * np.cos(lon_rad), -np.sin(lat_rad) * np.sin(lon_rad), np.cos(lat_rad)],
        [np.cos(lat_rad) * np.cos(lon_rad), np.cos(lat_rad) * np.sin(lon_rad), np.sin(lat_rad)]
    ])

    e, n, u = R @ np.array([dx, dy, dz])
    print(f"Satellite position relative to station: e={e}, n={n}, u={u}")  
    
    azimuth = np.arctan2(u, np.sqrt(e**2 + n**2))
    elevation = np.arctan2(n, e)

    return azimuth, elevation

In [61]:
satellite_position = (17487292.829, 6573938.932, 17927274.429)

# Convert DMS (degrees, minutes, seconds) to decimal degrees
def dms_to_decimal(degrees, minutes, seconds):
    return degrees + minutes/60 + seconds/3600

latitude = dms_to_decimal(44, 3, 48.114)
longitude = dms_to_decimal(7, 39, 40.605)

station_geodetic = (latitude, longitude)

In [62]:
x, y, z = compute_cartesian_coordinates(station_geodetic)
az, el = compute_azimuth_elevation(satellite_position, station_geodetic)

print(f"Cartesian coordinates of the GNSS station: x={x}, y={y}, z={z}")
print(f"Azimuth: {np.degrees(az)}, Elevation: {np.degrees(el)}")

Original geodetic coordinates: (44.063365, 7.661279166666667)
Converted to radians: latitude=0.7690507987580312, longitude=0.13371454637278074
Eccentricity squared: 0.0066943799901413165
Computed W: 0.9983798004667108
Eccentricity squared: 0.0066943799901413165
Computed W: 0.9983798004667108
Satellite position relative to station: e=4183913.143250176, n=240849.94694518886, u=19183242.86616875
Cartesian coordinates of the GNSS station: x=4549604.963101611, y=612000.1540939753, z=4413153.538512697
Azimuth: 77.67657533722381, Elevation: 3.294636393050315


**Ex 2**

In [64]:
satellite_coordinates = np.array([
    [22504974.806, 13900127.123, -2557240.727],
    [-3760396.280, -17947593.853, 19494169.070],
    [9355256.428, -12616043.006, 21189549.365],
    [23959436.524, 5078878.903, -10562274.680],
    [10228692.060, -19322124.315, 14550804.347],
    [23867142.480, -3892848.382, 10941892.224],
    [21493427.163, -15051899.636, 3348924.156],
    [14198354.868, 13792955.212, 17579451.054],
    [18493109.722, 4172695.812, 18776775.463],
    [-8106932.299, 12484531.565, 22195338.169],
    [8363810.808, 21755378.568, 13378858.106]
])

latitude = dms_to_decimal(43, 3, 48)
longitude = dms_to_decimal(7, 28, 41)

station_geodetic = (latitude, longitude)

# compute the cartesian coordinates of the GNSS station for each satellite
z, y, x = compute_cartesian_coordinates(station_geodetic)
print(f"Cartesian coordinates of the GNSS station for each satellite: x={x}, y={y}, z={z}")
# calculate pseudoranges
pseudoranges = np.sqrt(np.sum(((satellite_coordinates - np.array([x, y, z]).reshape(1, 3))**2), axis=1))
print("Pseudoranges for each satellite:", pseudoranges)
# calculate the D matrix
def compute_d_matrix(satellite_coordinates, station_cartesian):
    D = np.zeros((len(satellite_coordinates), 4))
    for i, sat in enumerate(satellite_coordinates):
        dx = sat[0] - station_cartesian[0]
        dy = sat[1] - station_cartesian[1]
        dz = sat[2] - station_cartesian[2]
        norm = np.sqrt(dx**2 + dy**2 + dz**2)
        D[i, 0] = dx / norm
        D[i, 1] = dy / norm
        D[i, 2] = dz / norm
        D[i, 3] = -1
    return D

D_matrix = compute_d_matrix(satellite_coordinates, np.array([x, y, z]))

Q_xx = np.linalg.inv(D_matrix.T @ D_matrix)

Q_xx_star = Q_xx[0:3, 0:3]

# radians conversion
lat_rad = np.radians(latitude)
lon_rad = np.radians(longitude)

R = np.array([
        [-np.sin(lon_rad), np.cos(lon_rad), 0],
        [-np.sin(lat_rad) * np.cos(lon_rad), -np.sin(lat_rad) * np.sin(lon_rad), np.cos(lat_rad)],
        [np.cos(lat_rad) * np.cos(lon_rad), np.cos(lat_rad) * np.sin(lon_rad), np.sin(lat_rad)]
    ])

Q_uu_star = R @ Q_xx_star @ R.T

# add a new column and a new row to Q_uu_star to make it a 4x4 matrix
Q_uu = np.zeros((4, 4))
Q_uu[:3, :3] = Q_uu_star
Q_uu[:, 3] = Q_xx[:, 3]
Q_uu[3, :] = Q_xx[3, :]

HDOP = np.sqrt((Q_uu[0, 0])**2 + (Q_uu[1, 1])**2)
PDOP = np.sqrt((Q_uu[0, 0])**2 + (Q_uu[1, 1])**2 + (Q_uu[2, 2])**2)
GDOP = np.sqrt((Q_uu[0, 0])**2 + (Q_uu[1, 1])**2 + (Q_uu[2, 2])**2 + Q_uu[3, 3]**2)
print(f"HDOP: {HDOP}, PDOP: {PDOP}, GDOP: {GDOP}")


Original geodetic coordinates: (43.06333333333333, 7.478055555555556)
Converted to radians: latitude=0.7515969535504914, longitude=0.1305166910914982
Eccentricity squared: 0.0066943799901413165
Computed W: 0.9984382383679747
Cartesian coordinates of the GNSS station for each satellite: x=4332644.846639261, y=607413.2929009288, z=4627456.834343379
Pseudoranges for each satellite: [23633656.0051288  25115826.43592942 21780480.54061694 25217710.04129146
 23030906.796387   21017160.77924196 23266739.71411455 20951012.00877693
 20333057.38403765 24585350.7723668  23239487.05334054]
HDOP: 0.3948632810412777, PDOP: 1.1984000094885698, GDOP: 1.2544138063517636
